In [48]:
import os
from dotenv import load_dotenv
from mp_api.client import MPRester
from pathlib import Path
from datetime import datetime

load_dotenv()
api_key = os.environ["MP_API_KEY"]

PATH = Path.cwd() / "reference_data" / "materials_project_reference_data"
PATH.mkdir(parents=True, exist_ok=True)

material = "K3MnCrO8" 
material_id = "mp-757140"




In [49]:
import json
from pathlib import Path

LOG_FILE = PATH / "mp_pull_log.json"
LOOKUP_FILE = PATH / "phonon_mp_lookup.json"

try:
    with MPRester(api_key) as mpr:
        phonon_doc = mpr.materials.phonon.search(material_id)

        # In addition, you can specfiy the phonon calculation methodology by using `phonon_method`,
        # the default is `None` so all results for mp-19907 are returned. Other options: "dfpt" or "pheasy"
        phonon_doc = mpr.materials.phonon.search(
            material_ids=[material_id], phonon_method="pheasy"
        )
except Exception as e:
    print(f"Error fetching phonon data: {e}")


# Load the existing log (or start a new one)
if LOG_FILE.exists():
    with open(LOG_FILE) as f:
        pull_log = json.load(f)
else:
    pull_log = {}

# Load the existing lookup file
if LOOKUP_FILE.exists():
    with open(LOOKUP_FILE) as f:
        lookup_log = json.load(f)
else:
    lookup_log = {}


# Add / update this material's entry into log file
pull_log[material_id] = {
    "material": material,
    "last_pulled": datetime.now().isoformat(timespec="seconds"),
}

with open(LOG_FILE, "w") as f:
    json.dump(pull_log, f, indent=4)

# Add / update this material's entry into lookup file
lookup_log[material_id] = {
    "material": material,
}
with open(LOOKUP_FILE, "w") as f:
    json.dump(lookup_log, f, indent=4)    
print(phonon_doc)

Output()

Output()

✓ Retrieved 1 SummaryDoc documents in 2.0s

Output()

✓ Retrieved 1 PhononBSDOSTask documents in 0.2s

[MPDataDoc<PhononBSDOSTask>(
sum_rules_breaking=None,
formula_anonymous='ABC3D8',
chemsys='Cr-K-Mn-O',
volume=202.13876890738356,
identifier=AlphaID(aaaddllu),
composition=Composition('K3 Mn1 Cr1 O8'),
phonon_method=<PhononMethod.PHEASY: 'pheasy'>,
supercell_matrix=((2.0, 0.0, 0.0), (0.0, 2.0, 0.0), (0.0, 0.0, 2.0)),
calc_meta=[CalcMeta(identifier=None, dir_name='/global/cfs/cdirs/matgen/jzhengap/block_2025-02-11-17-09-26-214067/launcher_2025-03-02-10-05-44-439299/launcher_2025-03-10-05-23-49-310914', name='generate_frequencies_eigenvectors--158096-0', parents=None), CalcMeta(identifier=None, dir_name='/global/cfs/cdirs/matgen/jzhengap/block_2025-02-11-17-09-26-214067/launcher_2025-03-02-10-06-12-417621/launcher_2025-03-10-01-33-55-885457', name='run_phonon_displacements--158097-0', parents=None), CalcMeta(identifier=None, dir_name='/global/cfs/cdirs/matgen/jzhengap/block_2025-02-11-17-09-26-214067/launcher_2025-03-02-10-05-44-439299/launcher_2025-03-10-00-15-02-210023', name='dielectr

In [50]:
# Bandstructures
import json

# Set the file save directories
SAVE_PATH = Path.cwd() / "reference_data" / "materials_project_reference_data" / "band_structure"
SAVE_PATH.mkdir(parents=True, exist_ok=True)

# Set the output file name and save location
output_fname = SAVE_PATH / f"{material}-phonondata.json"

# Execute fetch phonon data
try:
    with MPRester(api_key) as mpr:
        # Bandstructure path convention can be specified by using `path_type`. Default is "setyawan_curtarolo".
        phonon_bs = mpr.materials.phonon.get_bandstructure_from_material_id(
        material_id,
            phonon_method="pheasy",
            path_type="latimer_munro",
        )
        if not phonon_bs:
             raise ValueError(f"No phonon data found for material ID: {material_id}")

        phonon_dict = phonon_bs.to_pmg.as_dict()


        with open(output_fname, "w") as json_file:
            json.dump(phonon_dict, json_file, indent=4)
        print(f"Phonon data successfully saved to {output_fname}")

except Exception as e:
    print(f"Error fetching phonon data: {e}")
    


Output()

Phonon data successfully saved to c:\Users\sonny\OneDrive - University of Leeds\year 5\Y5 project\Code\Computational-methods-for-materials-discovery\reference_data\materials_project_reference_data\band_structure\K3MnCrO8-phonondata.json


In [51]:
# Force Constants (currently only available for `phonon_method="pheasy"`)
import numpy as np
from phonopy.file_IO import write_FORCE_CONSTANTS

# Set the file save directories
SAVE_PATH = Path.cwd() / "reference_data" / "materials_project_reference_data" / "force_constants"
SAVE_PATH.mkdir(parents=True, exist_ok=True)

# Set the output file name and save location
output_fname = SAVE_PATH / f"{material}_force_constants"

# Execute fetch phonon daya
try:
    with MPRester(api_key) as mpr:
        phonon_force_constants = mpr.materials.phonon.get_forceconstants_from_material_id(
            material_id, phonon_method="pheasy"
        )
    fc = np.array(phonon_force_constants)
    print(fc.shape)
    write_FORCE_CONSTANTS(fc, filename=str(SAVE_PATH / output_fname))
    print(f"Force constants successfully saved to {output_fname}")

except Exception as e:
    print(f"Error fetching force constant data {e}")



Output()

(104, 104, 3, 3)
Force constants successfully saved to c:\Users\sonny\OneDrive - University of Leeds\year 5\Y5 project\Code\Computational-methods-for-materials-discovery\reference_data\materials_project_reference_data\force_constants\K3MnCrO8_force_constants


In [52]:
# DOS
import json
from pathlib import Path
from monty.json import MontyEncoder
from mp_api.client import MPRester

# Set the file save directories
SAVE_PATH = Path.cwd() / "reference_data" / "materials_project_reference_data" / "dos"
SAVE_PATH.mkdir(parents=True, exist_ok=True)
output_fname = SAVE_PATH / f"{material}_phonondos.json"

with MPRester(api_key) as mpr:
    try:
        phonon_dos = mpr.materials.phonon.get_dos_from_material_id(
            material_id, phonon_method="pheasy"
        )
        dos_pmg = phonon_dos.to_pmg
        dos_dict = dos_pmg.as_dict()
        with open(output_fname, "w") as f:
            json.dump(dos_dict, f, cls = MontyEncoder, indent =2 )
        print(f"DOS data successfully saved to {output_fname}")
        
    except Exception as e:
        print(f"Error fetching DOS data {e}")




Output()

DOS data successfully saved to c:\Users\sonny\OneDrive - University of Leeds\year 5\Y5 project\Code\Computational-methods-for-materials-discovery\reference_data\materials_project_reference_data\dos\K3MnCrO8_phonondos.json


In [53]:
# Phonopy settings: the structure, supercell and primitive matrices that go with
# the force constants, plus Born charges and dielectric tensors for the NAC
import json
from pathlib import Path
from mp_api.client import MPRester

SAVE_PATH = Path.cwd() / "reference_data" / "materials_project_reference_data" / "phonopy_settings"
SAVE_PATH.mkdir(parents=True, exist_ok=True)
output_fname = SAVE_PATH / f"{material}_phonopy_settings.json"

def to_list(value):
    """Turns tuples / numpy arrays into plain lists so they can be saved as JSON."""
    if value is None:
        return None
    return [to_list(item) if hasattr(item, "__len__") else float(item) for item in value]

try:
    with MPRester(api_key) as mpr:
        docs = mpr.materials.phonon.search(material_ids=[material_id], phonon_method="pheasy")
    if not docs:
        raise ValueError(f"No pheasy phonon document found for {material_id}")
    doc = docs[0]

    structure = doc.structure
    settings = {
        "material_id": material_id,
        "material": material,
        "phonon_method": str(doc.phonon_method),
        "code": doc.code,
        # The unit cell the supercell matrix and force constants refer to
        "structure": structure.as_dict() if hasattr(structure, "as_dict") else structure,
        "supercell_matrix": to_list(doc.supercell_matrix),
        "primitive_matrix": to_list(doc.primitive_matrix),
        # Born charges are for symmetrically inequivalent atoms only (same as a phonopy BORN file)
        "born": to_list(doc.born),
        "epsilon_static": to_list(doc.epsilon_static),
        "epsilon_electronic": to_list(doc.epsilon_electronic),
    }

    with open(output_fname, "w") as json_file:
        json.dump(settings, json_file, indent=4)

    print(f"Saved to {output_fname}")
    print(f"atoms in unit cell: {len(settings['structure']['sites'])}")
    print(f"supercell matrix: {settings['supercell_matrix']}")
    print(f"Born charges saved: {settings['born'] is not None}")

except Exception as e:
    print(f"Error fetching phonopy settings: {e}")

Output()

✓ Retrieved 1 SummaryDoc documents in 2.1s

Output()

✓ Retrieved 1 PhononBSDOSTask documents in 0.2s

Saved to c:\Users\sonny\OneDrive - University of Leeds\year 5\Y5 project\Code\Computational-methods-for-materials-discovery\reference_data\materials_project_reference_data\phonopy_settings\K3MnCrO8_phonopy_settings.json
atoms in unit cell: 13
supercell matrix: [[2.0, 0.0, 0.0], [0.0, 2.0, 0.0], [0.0, 0.0, 2.0]]
Born charges saved: True


In [54]:
from pathlib import Path
from mp_api.client import MPRester

SAVE_PATH = Path.cwd() / "materials_cif" 
SAVE_PATH.mkdir(parents=True, exist_ok=True)
output_fname = SAVE_PATH / f"{material}.cif"

try:
    with MPRester(api_key) as mpr:
        # Fetch the pymatgen Structure object
        structure = mpr.get_structure_by_material_id(material_id)
    
        # Export and save directly to a CIF file
        structure.to(filename=output_fname)
        print(f"Saved {output_fname} successfully.")
except Exception as e:
    print(f"Error fetching struture .cif data: {e}")

Output()

✓ Retrieved 1 MaterialsDoc documents in 0.7s

Saved c:\Users\sonny\OneDrive - University of Leeds\year 5\Y5 project\Code\Computational-methods-for-materials-discovery\materials_cif\K3MnCrO8.cif successfully.
